# NB 04 — Generación de Asientos y Salida Excel
**Proyecto 5 · Automatización Contable**

**Input:** `data/processed/facturas_clasificadas.json`  
**Template:** `config/Diario_2025_plantilla.xlsx`  
**Output:**
- `data/output/Diario_{periodo}.xlsx` — Libro Diario con asientos nuevos
- `data/output/reporte_revision_{fecha}.xlsx` — Facturas que Carlos debe revisar

### Estructura de asientos
**COMPRA** (5 filas): gasto + IGV crédito + prov. por pagar + clase 9 + clase 79  
**VENTA** (3 filas): cuentas por cobrar + IGV débito + ingresos por servicios

## 0. Setup

In [1]:
import json
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pathlib import Path
from datetime import datetime, date
from copy import copy
import cliente_config

BASE_DIR       = Path('../')
CONFIG_DIR     = BASE_DIR / 'config'
INPUT_PATH     = BASE_DIR / 'data/processed/facturas_clasificadas.json'
OUTPUT_DIR     = BASE_DIR / 'data/output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cliente activo — debe coincidir con el usado en NB03 para este lote
CLIENTE_ID = 'carlos_torres'
CLIENTE       = cliente_config.cargar_cliente(CLIENTE_ID, CONFIG_DIR)
PLAN_CONTABLE = CLIENTE['plan']
CODIGOS       = CLIENTE['codigos_estructurales']
TEMPLATE_PATH = Path(CLIENTE['template_diario'])

# Fills para overlay de confianza (se aplican encima del estilo base del template)
FILL_AMARILLO = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
FILL_NARANJA  = PatternFill(start_color='FFD580', end_color='FFD580', fill_type='solid')

print(f'Cliente activo: {CLIENTE["nombre"]} ({CLIENTE["rubro"]})')
print('Setup OK')
print(f'Template: {TEMPLATE_PATH}')


Cliente activo: CT PRIME CONSULTING SAC (libros de Contacto Creativo TI S.A.C.) (servicios)
Setup OK
Template: D:\Proyecto_Gabriel\01_Portfolio\Proyecto_contable\Data\libro_diario_piloto.xlsx


## 1. Funciones auxiliares

In [2]:
def parse_fecha(fecha_str) -> datetime:
    """Convierte string YYYY-MM-DD a datetime. Retorna hoy si falla."""
    if isinstance(fecha_str, datetime):
        return fecha_str
    if isinstance(fecha_str, date):
        return datetime(fecha_str.year, fecha_str.month, fecha_str.day)
    try:
        return datetime.strptime(str(fecha_str)[:10], '%Y-%m-%d')
    except Exception:
        return datetime.today()


def round2(val) -> float:
    """Redondeo a 2 decimales seguro."""
    try:
        return round(float(val or 0), 2)
    except (ValueError, TypeError):
        return 0.0


def get_ultimo_nro(ws) -> int:
    """Lee el mayor número de asiento existente en la columna A."""
    max_nro = 0
    for row in ws.iter_rows(min_row=6, values_only=True):
        val = row[0]
        if isinstance(val, (int, float)) and val > max_nro:
            max_nro = int(val)
    return max_nro


def get_ultima_fila_con_datos(ws) -> int:
    """Retorna el índice de la última fila con datos (para saber dónde insertar)."""
    ultima = 5
    for row in ws.iter_rows(min_row=6):
        if any(cell.value is not None and cell.value != '' for cell in row):
            ultima = row[0].row
    return ultima


def _extraer_estilos_de_fila(row) -> dict:
    return {
        cell.column: {
            'font':          copy(cell.font),
            'border':        copy(cell.border),
            'alignment':     copy(cell.alignment),
            'number_format': cell.number_format,
        }
        for cell in row
    }


def leer_estilos_referencia(ws) -> dict:
    """Lee estilos de la primera fila de datos para heredarlos en filas nuevas.
    Fallback a fila de headers (fila 5) si el template está vacío (primer lote del piloto)."""
    for row in ws.iter_rows(min_row=6):
        if any(c.value is not None for c in row):
            return _extraer_estilos_de_fila(row)
    header_row = list(ws.iter_rows(min_row=5, max_row=5))[0]
    return _extraer_estilos_de_fila(header_row)


def leer_docs_existentes(ws) -> set:
    """Lee todos los nro_doc ya registrados en columna J (col 10) del Diario."""
    existentes = set()
    for row in ws.iter_rows(min_row=6, min_col=10, max_col=10, values_only=True):
        val = row[0]
        if val:
            existentes.add(str(val).strip())
    return existentes


def aplicar_estilo_fila(ws, fila: int, confianza: float):
    """Colorea la fila según nivel de confianza."""
    if confianza < 0.70:
        fill = FILL_AMARILLO
    elif confianza < 0.90:
        fill = FILL_NARANJA
    else:
        fill = None

    if fill:
        for col in range(1, 11):
            ws.cell(row=fila, column=col).fill = fill


def escribir_celda(ws, fila: int, col: int, valor, ref_estilos: dict, number_format: str = None):
    cell = ws.cell(row=fila, column=col, value=valor)
    est = ref_estilos.get(col, {})
    if est.get('font'):      cell.font      = copy(est['font'])
    if est.get('border'):    cell.border    = copy(est['border'])
    if est.get('alignment'): cell.alignment = copy(est['alignment'])
    cell.number_format = number_format or est.get('number_format', 'General')
    return cell

print('Auxiliares OK')

Auxiliares OK


In [3]:
def escribir_control_facturas(wb, facturas_todas: list[dict]) -> None:
    """
    Crea (o sobreescribe) la hoja Control_Facturas.
    Una fila por factura, ordenada por fecha_emision.
    Cols 1-12: auto-pobladas | Cols 13-18: llenado manual (fondo amarillo claro).
    NC/ND: col 18 (OBSERVACIONES) pre-populada con doc_referencia.
    """
    HEADERS_AUTO = [
        'TIPO_OPERACION', 'FECHA_EMISION', 'RUC_TERCERO', 'RAZON_SOCIAL',
        'NRO_DOCUMENTO', 'DESCRIPCION_SERVICIO', 'BASE_IMPONIBLE', 'IGV',
        'TOTAL', 'MONEDA', 'TIENE_DETRACCION', 'MONTO_DETRACCION',
    ]
    HEADERS_MANUAL = [
        'ESTADO_PAGO', 'FECHA_VCTO', 'BANCO', 'MONTO_PAGADO', 'NRO_OPERACION', 'OBSERVACIONES',
    ]
    HEADERS = HEADERS_AUTO + HEADERS_MANUAL
    N_AUTO  = len(HEADERS_AUTO)

    if 'Control_Facturas' in wb.sheetnames:
        del wb['Control_Facturas']
    ws = wb.create_sheet('Control_Facturas')

    fill_header = PatternFill(start_color='0A1A3F', end_color='0A1A3F', fill_type='solid')
    font_header = Font(bold=True, name='Calibri', size=10, color='C9A227')
    align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    fill_manual  = PatternFill(start_color='FFF9C4', end_color='FFF9C4', fill_type='solid')
    thin = Side(style='thin')
    border_data = Border(left=thin, right=thin, top=thin, bottom=thin)
    font_data = Font(name='Calibri', size=10)

    # Fila de headers
    for col, h in enumerate(HEADERS, 1):
        cell = ws.cell(row=1, column=col, value=h)
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = align_center

    # Filas de datos — una por factura
    facturas_ordenadas = sorted(facturas_todas, key=lambda x: x.get('fecha_emision', ''))

    for i, f in enumerate(facturas_ordenadas, start=2):
        tipo     = f.get('tipo_operacion', '')
        tipo_doc = f.get('tipo_doc', 'FACTURA')

        if tipo == 'COMPRA':
            ruc_tercero  = f.get('ruc_emisor', '')
            razon_social = f.get('razon_social_emisor', '')
        else:
            ruc_tercero  = f.get('ruc_receptor', '')
            razon_social = f.get('razon_social_receptor', '')

        serie   = f.get('serie', '')
        numero  = f.get('numero', '')
        nro_doc = f'{serie}-{numero}' if serie else numero

        tiene_det = 'SI' if f.get('tiene_detraccion') else 'NO'
        monto_det = round2(f.get('monto_detraccion') or 0)

        valores_auto = [
            tipo,
            parse_fecha(f.get('fecha_emision')),
            ruc_tercero,
            razon_social,
            nro_doc,
            f.get('descripcion_servicio', ''),
            round2(f.get('base_imponible')),
            round2(f.get('igv')),
            round2(f.get('total')),
            f.get('moneda', 'PEN'),
            tiene_det,
            monto_det,
        ]

        for col, val in enumerate(valores_auto, 1):
            cell = ws.cell(row=i, column=col, value=val)
            cell.font = font_data
            cell.border = border_data
            if col in (7, 8, 9, 12):   # montos
                cell.number_format = '#,##0.00'
            elif col == 2:              # fecha
                cell.number_format = 'DD/MM/YYYY'

        # Columnas manuales — fondo amarillo claro
        # Col 18 (OBSERVACIONES): pre-poblar con doc_referencia para NC/ND
        obs_valor = None
        if tipo_doc in ('NOTA_CREDITO', 'NOTA_DEBITO'):
            doc_ref = f.get('doc_referencia', '')
            obs_valor = f'Ref: {doc_ref}' if doc_ref else f'{tipo_doc}'

        for col in range(N_AUTO + 1, len(HEADERS) + 1):
            val = obs_valor if col == 18 else None
            cell = ws.cell(row=i, column=col, value=val)
            cell.fill = fill_manual
            cell.font = font_data
            cell.border = border_data
            if col == 16:               # MONTO_PAGADO
                cell.number_format = '#,##0.00'

    # Anchos de columna
    anchos = {
        'A': 14, 'B': 14, 'C': 14, 'D': 35, 'E': 20, 'F': 45,
        'G': 14, 'H': 12, 'I': 14, 'J': 8,  'K': 16, 'L': 16,
        'M': 14, 'N': 14, 'O': 14, 'P': 14, 'Q': 16, 'R': 20,
    }
    for col_letter, width in anchos.items():
        ws.column_dimensions[col_letter].width = width

    ws.row_dimensions[1].height = 30
    print(f'  Control_Facturas: {len(facturas_ordenadas)} facturas escritas')

print('escribir_control_facturas OK')

escribir_control_facturas OK


## 2. Generadores de asientos

In [4]:
def asiento_compra(factura: dict, nro: int) -> list[dict]:
    """
    Genera bloque de 5 filas para una factura de COMPRA.
    Estructura:
      [0] cuenta_gasto (6XXXXXX) DEBE = base_imponible
      [1] IGV credito fiscal DEBE = igv
      [2] cuentas por pagar HABER = total
      [3] clase 9 (costo/gasto por funcion) DEBE = base_imponible
      [4] clase 79 (cargas imputables) HABER = base_imponible
    """
    fecha         = parse_fecha(factura.get('fecha_emision'))
    serie         = factura.get('serie', '')
    numero        = factura.get('numero', '')
    nro_doc       = f'{serie}-{numero}' if serie else numero
    razon         = factura.get('razon_social_emisor', '')
    glosa         = f'Reg. Compras {nro_doc}'
    base          = round2(factura.get('base_imponible'))
    igv           = round2(factura.get('igv'))
    total         = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv  = round2(total - base)

    cuenta_gasto = factura.get('cuenta_debe') or CODIGOS.get('cuenta_gasto_default')
    confianza    = factura.get('confianza_clasificacion', 0.70)
    if not cuenta_gasto:
        # Rubro sin cuenta de gasto por defecto (ej. retail) — no adivinar, forzar revisión
        cuenta_gasto = 'SIN_CLASIFICAR'
        confianza = 0.0
    detalle_gasto = factura.get('detalle_cuenta') or PLAN_CONTABLE['cuentas'].get(cuenta_gasto, '')

    cuenta_igv = CODIGOS['igv']
    cuenta_cxp = CODIGOS['cuentas_por_pagar'] or 'SIN_CONFIGURAR_CXP'
    cuenta_c9  = CODIGOS['clase_9']
    cuenta_c79 = CODIGOS['clase_79']

    if not CODIGOS['cuentas_por_pagar']:
        # Codigo estructural de CxP no configurado para este cliente/rubro — forzar revision
        confianza = 0.0

    rows = [
        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_gasto, 'detalle': detalle_gasto,
         'debe': base, 'haber': None,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_igv, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_igv, ''),
         'debe': igv, 'haber': None,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_cxp, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_cxp, ''),
         'debe': None, 'haber': total,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_c9, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_c9, ''),
         'debe': base, 'haber': None,
         'cliente_proveedor': None, 'nro_doc': None, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_c79, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_c79, ''),
         'debe': None, 'haber': base,
         'cliente_proveedor': None, 'nro_doc': None, 'confianza': confianza},
    ]
    return rows


def asiento_venta(factura: dict, nro: int) -> list[dict]:
    """
    Genera bloque de 3 filas para una factura de VENTA.
    Estructura:
      [0] cuentas por cobrar DEBE = total
      [1] IGV debito fiscal HABER = igv
      [2] ingresos HABER = base_imponible
    """
    fecha   = parse_fecha(factura.get('fecha_emision'))
    serie   = factura.get('serie', '')
    numero  = factura.get('numero', '')
    nro_doc = f'{serie}-{numero}' if serie else numero
    razon   = factura.get('razon_social_receptor', '')
    glosa   = f'Reg. Ventas {nro_doc}'
    base    = round2(factura.get('base_imponible'))
    igv     = round2(factura.get('igv'))
    total   = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv  = round2(total - base)

    cuenta_igv = CODIGOS['igv']
    cuenta_cxc = CODIGOS['cuentas_por_cobrar'] or 'SIN_CONFIGURAR_CXC'
    cuenta_ing = CODIGOS['ingresos']
    confianza  = 1.0 if CODIGOS['cuentas_por_cobrar'] else 0.0

    rows = [
        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_cxc, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_cxc, ''),
         'debe': total, 'haber': None,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_igv, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_igv, ''),
         'debe': None, 'haber': igv,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_ing, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_ing, ''),
         'debe': None, 'haber': base,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},
    ]
    return rows

print('Generadores OK')

Generadores OK


In [5]:
TIPOS_AJUSTE = {'NOTA_CREDITO', 'NOTA_DEBITO'}


def asiento_nc_venta(factura: dict, nro: int) -> list[dict]:
    """
    Asiento inverso para NC emitida a un cliente (reversa de venta).
    Estructura (3 filas, cuadre: igv + base = total):
      [0] IGV DEBE = igv
      [1] Ingresos DEBE = base_imponible
      [2] CxCobrar HABER = total
    """
    fecha     = parse_fecha(factura.get('fecha_emision'))
    serie     = factura.get('serie', '')
    numero    = factura.get('numero', '')
    doc_ref   = factura.get('doc_referencia', '')
    nro_doc   = f'{serie}-{numero}' if serie else numero
    razon     = factura.get('razon_social_receptor', '')
    glosa     = f'NC {nro_doc} ref: {doc_ref}'
    base      = round2(factura.get('base_imponible'))
    igv       = round2(factura.get('igv'))
    total     = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv  = round2(total - base)

    cuenta_igv = CODIGOS['igv']
    cuenta_ing = CODIGOS['ingresos']
    cuenta_cxc = CODIGOS['cuentas_por_cobrar'] or 'SIN_CONFIGURAR_CXC'
    confianza  = 1.0 if CODIGOS['cuentas_por_cobrar'] else 0.0

    return [
        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_igv, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_igv, ''),
         'debe': igv, 'haber': None,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_ing, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_ing, ''),
         'debe': base, 'haber': None,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_cxc, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_cxc, ''),
         'debe': None, 'haber': total,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},
    ]


def asiento_nc_compra(factura: dict, nro: int) -> list[dict]:
    """
    Asiento para NC recibida de proveedor (reduce CxPagar y reversa gasto).
    Estructura (5 filas, cuadre: total + base = igv + base + base → total = igv + base ✓):
      [0] CxPagar   DEBE  = total
      [1] IGV        HABER = igv
      [2] Cuenta gasto HABER = base_imponible
      [3] Clase 79   DEBE  = base_imponible
      [4] Clase 9     HABER = base_imponible
    """
    fecha         = parse_fecha(factura.get('fecha_emision'))
    serie         = factura.get('serie', '')
    numero        = factura.get('numero', '')
    doc_ref       = factura.get('doc_referencia', '')
    nro_doc       = f'{serie}-{numero}' if serie else numero
    razon         = factura.get('razon_social_emisor', '')
    glosa         = f'NC recibida {nro_doc} ref: {doc_ref}'
    base          = round2(factura.get('base_imponible'))
    igv           = round2(factura.get('igv'))
    total         = round2(factura.get('total'))
    # Normalizar: si extracción fue inconsistente, recalcular desde total con IGV 18%
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv  = round2(total - base)

    cuenta_gasto  = factura.get('cuenta_debe') or CODIGOS.get('cuenta_gasto_default') or 'SIN_CLASIFICAR'
    detalle_gasto = factura.get('detalle_cuenta') or PLAN_CONTABLE['cuentas'].get(cuenta_gasto, '')
    cuenta_igv    = CODIGOS['igv']
    cuenta_cxp    = CODIGOS['cuentas_por_pagar'] or 'SIN_CONFIGURAR_CXP'
    cuenta_c9     = CODIGOS['clase_9']
    cuenta_c79    = CODIGOS['clase_79']
    confianza     = 1.0 if CODIGOS['cuentas_por_pagar'] else 0.0

    return [
        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_cxp, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_cxp, ''),
         'debe': total, 'haber': None,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_igv, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_igv, ''),
         'debe': None, 'haber': igv,
         'cliente_proveedor': razon, 'nro_doc': nro_doc, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_gasto, 'detalle': detalle_gasto,
         'debe': None, 'haber': base,
         'cliente_proveedor': None, 'nro_doc': None, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_c79, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_c79, ''),
         'debe': base, 'haber': None,
         'cliente_proveedor': None, 'nro_doc': None, 'confianza': confianza},

        {'nro': nro, 'fecha': fecha, 'glosa': glosa, 'obs': None,
         'cuenta': cuenta_c9, 'detalle': PLAN_CONTABLE['cuentas'].get(cuenta_c9, ''),
         'debe': None, 'haber': base,
         'cliente_proveedor': None, 'nro_doc': None, 'confianza': confianza},
    ]

print('Generadores NC/ND OK')

Generadores NC/ND OK


## 3. Escritura al Excel

In [6]:
def escribir_asiento_en_excel(ws, fila_inicio: int, filas_asiento: list[dict], ref_estilos: dict) -> int:
    """
    Escribe un bloque de filas de asiento en el Excel.
    Retorna la siguiente fila disponible.
    """
    for i, fila in enumerate(filas_asiento):
        fila_excel = fila_inicio + i
        confianza = fila.get('confianza', 1.0)

        escribir_celda(ws, fila_excel, 1,  fila['nro'],                    ref_estilos)                          # A: NRO
        escribir_celda(ws, fila_excel, 2,  fila['fecha'],                  ref_estilos, 'DD/MM/YYYY')            # B: FECHA
        escribir_celda(ws, fila_excel, 3,  fila['glosa'],                  ref_estilos)                          # C: GLOSA
        escribir_celda(ws, fila_excel, 4,  fila['obs'],                    ref_estilos)                          # D: OBS
        escribir_celda(ws, fila_excel, 5,  fila['cuenta'],                 ref_estilos)                          # E: CUENTA
        escribir_celda(ws, fila_excel, 6,  fila['detalle'],                ref_estilos)                          # F: DETALLE
        escribir_celda(ws, fila_excel, 7,  fila['debe'],   ref_estilos, number_format='#,##0.00')               # G: DEBE
        escribir_celda(ws, fila_excel, 8,  fila['haber'],  ref_estilos, number_format='#,##0.00')               # H: HABER
        escribir_celda(ws, fila_excel, 9,  fila.get('cliente_proveedor'),  ref_estilos)                          # I: CLIENTE/PROVEEDOR
        escribir_celda(ws, fila_excel, 10, fila.get('nro_doc'),            ref_estilos)                          # J: NRO. DOCUMENTO

        aplicar_estilo_fila(ws, fila_excel, confianza)

    # Fila separadora en blanco entre asientos
    fila_sep = fila_inicio + len(filas_asiento)
    ws.cell(row=fila_sep, column=1).value = None

    return fila_sep + 1

print('Escritura OK')

Escritura OK


## 4. Pipeline principal

In [7]:
with open(INPUT_PATH, encoding='utf-8') as f:
    facturas = json.load(f)

# Pre-enriquecimiento: llenar razon_social vacíos usando otras facturas del mismo RUC
# Caso típico: E001-30 no tiene razon_social_receptor, pero E001-31 sí tiene el mismo receptor
ruc_to_razon = {}
for f in facturas:
    for ruc_f, nombre_f in [('ruc_receptor', 'razon_social_receptor'),
                              ('ruc_emisor',   'razon_social_emisor')]:
        ruc_val    = str(f.get(ruc_f,    '') or '').strip()
        nombre_val = str(f.get(nombre_f, '') or '').strip()
        if ruc_val and nombre_val and nombre_val.lower() not in ('null', 'none'):
            ruc_to_razon[ruc_val] = nombre_val

enriquecidos = 0
for f in facturas:
    for ruc_f, nombre_f in [('ruc_receptor', 'razon_social_receptor'),
                              ('ruc_emisor',   'razon_social_emisor')]:
        if not str(f.get(nombre_f, '') or '').strip():
            ruc_val = str(f.get(ruc_f, '') or '').strip()
            if ruc_val in ruc_to_razon:
                f[nombre_f] = ruc_to_razon[ruc_val]
                enriquecidos += 1
                print(f'  Enriquecido: {f.get("archivo_origen","?")} → {nombre_f} = {ruc_to_razon[ruc_val]}')

if enriquecidos:
    print(f'  Total enriquecidos: {enriquecidos}')

# Separar y ordenar por fecha
compras = sorted([f for f in facturas if f.get('tipo_operacion') == 'COMPRA'],
                 key=lambda x: x.get('fecha_emision', ''))
ventas  = sorted([f for f in facturas if f.get('tipo_operacion') == 'VENTA'],
                 key=lambda x: x.get('fecha_emision', ''))

print(f'Facturas en JSON: {len(compras)} compras, {len(ventas)} ventas')

# Determinar período para el nombre del archivo
todas_fechas = [f.get('fecha_emision', '') for f in facturas if f.get('fecha_emision')]
periodo = datetime.now().strftime('%Y%m') if not todas_fechas else todas_fechas[0][:7].replace('-', '')
output_path = OUTPUT_DIR / f'Diario_{periodo}.xlsx'

# Cargar template y eliminar hojas genéricas vacías
wb = openpyxl.load_workbook(str(TEMPLATE_PATH))
for hoja_extra in ['Hoja1', 'Sheet1', 'Sheet']:
    if hoja_extra in wb.sheetnames:
        del wb[hoja_extra]
        print(f'  Hoja eliminada: {hoja_extra}')

ws = wb['Libro_diario']
REF_ESTILOS     = leer_estilos_referencia(ws)
DOCS_EXISTENTES = leer_docs_existentes(ws)

# Determinar punto de inserción y siguiente NRO
ultima_fila   = get_ultima_fila_con_datos(ws)
siguiente_nro = get_ultimo_nro(ws) + 1
fila_actual   = ultima_fila + 2

print(f'Última fila con datos: {ultima_fila}')
print(f'Primer NRO automático: {siguiente_nro}')
print(f'Inicio de escritura: fila {fila_actual}')
print(f'Documentos ya registrados en Diario: {len(DOCS_EXISTENTES)}')

# Contadores
facturas_para_reporte = []
asientos_escritos     = 0
duplicados_ignorados  = 0

# Escribir COMPRAS — tipo_doc se resuelve ANTES de la deduplicación
for factura in compras:
    tipo_doc = factura.get('tipo_doc', 'FACTURA')
    serie    = factura.get('serie', '')
    numero   = factura.get('numero', '')
    nro_doc  = f'{serie}-{numero}' if serie else numero

    if tipo_doc in TIPOS_AJUSTE:
        dedup_key = f'NC_{nro_doc}_{factura.get("doc_referencia", "")}'
    else:
        dedup_key = nro_doc

    if dedup_key in DOCS_EXISTENTES:
        print(f'  ⚠️ Duplicado ignorado (compra): {nro_doc}')
        duplicados_ignorados += 1
        continue

    if tipo_doc in TIPOS_AJUSTE:
        filas = asiento_nc_compra(factura, siguiente_nro)
        print(f'  NC/ND compra: {nro_doc} → asiento inverso (ref: {factura.get("doc_referencia","?")})')
    else:
        filas = asiento_compra(factura, siguiente_nro)

    fila_actual = escribir_asiento_en_excel(ws, fila_actual, filas, REF_ESTILOS)
    DOCS_EXISTENTES.add(dedup_key)

    confianza = factura.get('confianza_clasificacion', 0.70)
    if confianza < 0.90 or tipo_doc in TIPOS_AJUSTE:
        facturas_para_reporte.append(factura)

    asientos_escritos += 1
    siguiente_nro += 1

# Escribir VENTAS — tipo_doc se resuelve ANTES de la deduplicación
for factura in ventas:
    tipo_doc = factura.get('tipo_doc', 'FACTURA')
    serie    = factura.get('serie', '')
    numero   = factura.get('numero', '')
    nro_doc  = f'{serie}-{numero}' if serie else numero

    if tipo_doc in TIPOS_AJUSTE:
        dedup_key = f'NC_{nro_doc}_{factura.get("doc_referencia", "")}'
    else:
        dedup_key = nro_doc

    if dedup_key in DOCS_EXISTENTES:
        print(f'  ⚠️ Duplicado ignorado (venta): {nro_doc}')
        duplicados_ignorados += 1
        continue

    if tipo_doc in TIPOS_AJUSTE:
        filas = asiento_nc_venta(factura, siguiente_nro)
        print(f'  NC/ND venta: {nro_doc} → asiento inverso (ref: {factura.get("doc_referencia","?")})')
    else:
        filas = asiento_venta(factura, siguiente_nro)

    fila_actual = escribir_asiento_en_excel(ws, fila_actual, filas, REF_ESTILOS)
    DOCS_EXISTENTES.add(dedup_key)

    confianza = factura.get('confianza_clasificacion', 0.70)
    if confianza < 0.90 or tipo_doc in TIPOS_AJUSTE:
        facturas_para_reporte.append(factura)

    asientos_escritos += 1
    siguiente_nro += 1

# Ajustar anchos columna Libro_diario
ws.column_dimensions['A'].width = 6
ws.column_dimensions['B'].width = 12
ws.column_dimensions['C'].width = 35
ws.column_dimensions['D'].width = 20
ws.column_dimensions['E'].width = 12
ws.column_dimensions['F'].width = 55
ws.column_dimensions['G'].width = 14
ws.column_dimensions['H'].width = 14
ws.column_dimensions['I'].width = 35
ws.column_dimensions['J'].width = 22

# Escribir hoja Control_Facturas
escribir_control_facturas(wb, facturas)

# Guardar
wb.save(str(output_path))

# Segunda pasada: sobrescribir fórmulas TOTALES con rango explícito
# La tabla estructurada `tdiario` puede tener límites fijos — reemplazamos con SUM de rango dinámico
wb2 = openpyxl.load_workbook(str(output_path))
ws2 = wb2['Libro_diario']
ws2['G3'] = f'=SUM(G6:G{fila_actual})'
ws2['H3'] = f'=SUM(H6:H{fila_actual})'
ws2['H2'] = '=G3-H3'
wb2.save(str(output_path))

print(f'\nDiario guardado: {output_path}')
print(f'Asientos escritos: {asientos_escritos}')
if duplicados_ignorados:
    print(f'Duplicados ignorados: {duplicados_ignorados}')
print(f'Facturas para reporte de revisión: {len(facturas_para_reporte)}')
print(f'Fórmulas TOTALES actualizadas hasta fila {fila_actual}')

Facturas en JSON: 0 compras, 3 ventas
  Hoja eliminada: Hoja1
Última fila con datos: 5
Primer NRO automático: 1
Inicio de escritura: fila 7
Documentos ya registrados en Diario: 0
  NC/ND venta: E001-00000003 → asiento inverso (ref: E001-30)
  Control_Facturas: 3 facturas escritas



Diario guardado: ..\data\output\Diario_202502.xlsx
Asientos escritos: 3
Facturas para reporte de revisión: 1
Fórmulas TOTALES actualizadas hasta fila 19


## 5. Reporte de revisión para Carlos

In [8]:
def generar_reporte_revision(facturas_revision: list[dict], output_dir: Path) -> Path:
    """
    Genera un Excel con las facturas que Carlos debe revisar.
    Columna 'cuenta_final' (col 10) en verde claro — Carlos escribe la corrección ahí.
    NC/ND siempre incluidas. Incluye flags tributarios y doc_referencia.
    """
    fecha_str = datetime.now().strftime('%Y%m%d')
    rpt_path = output_dir / f'reporte_revision_{fecha_str}.xlsx'

    wb_rpt = openpyxl.Workbook()
    ws_rpt = wb_rpt.active
    ws_rpt.title = 'Revisión'

    headers = [
        'ruc_proveedor', 'razon_social', 'descripcion_servicio',
        'base_imponible', 'cuenta_sugerida', 'detalle_cuenta_sugerida',
        'confianza', 'origen', 'razon_sistema',
        'cuenta_final',          # ← Carlos llena esta columna (col 10)
        'archivo_origen',
        'requiere_bancarizacion',
        'otorga_credito_igv',
        'doc_referencia',
    ]

    for col, h in enumerate(headers, 1):
        cell = ws_rpt.cell(row=1, column=col, value=h.upper())
        cell.fill = PatternFill(start_color='0A1A3F', end_color='0A1A3F', fill_type='solid')
        cell.font = Font(bold=True, name='Calibri', size=10, color='C9A227')

    for row_idx, factura in enumerate(facturas_revision, 2):
        confianza = factura.get('confianza_clasificacion', 0)
        tipo_doc  = factura.get('tipo_doc', 'FACTURA')
        valores = [
            factura.get('ruc_emisor', ''),
            factura.get('razon_social_emisor', ''),
            factura.get('descripcion_servicio', ''),
            factura.get('base_imponible', 0),
            factura.get('cuenta_debe', ''),
            factura.get('detalle_cuenta', ''),
            confianza,
            factura.get('origen_clasificacion', ''),
            factura.get('razon_clasificacion', ''),
            '',  # cuenta_final — Carlos llena aquí
            factura.get('archivo_origen', ''),
            'SI' if factura.get('requiere_bancarizacion') else 'NO',
            'SI' if factura.get('otorga_credito_igv', True) else 'NO',
            factura.get('doc_referencia', ''),
        ]
        for col_idx, val in enumerate(valores, 1):
            cell = ws_rpt.cell(row=row_idx, column=col_idx, value=val)
            cell.font = Font(name='Calibri', size=10)
            # NC/ND siempre en naranja para distinguirlas visualmente
            if tipo_doc in ('NOTA_CREDITO', 'NOTA_DEBITO'):
                cell.fill = FILL_NARANJA
            elif confianza < 0.70:
                cell.fill = FILL_AMARILLO
            elif confianza < 0.90:
                cell.fill = FILL_NARANJA

    # Columna cuenta_final en verde claro
    fill_verde = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    for row in range(2, len(facturas_revision) + 2):
        ws_rpt.cell(row=row, column=10).fill = fill_verde

    anchos = {
        'A': 14, 'B': 35, 'C': 40, 'D': 14, 'E': 14, 'F': 50,
        'G': 10, 'H': 16, 'I': 45, 'J': 14, 'K': 25, 'L': 22, 'M': 18, 'N': 20,
    }
    for col_letter, width in anchos.items():
        ws_rpt.column_dimensions[col_letter].width = width

    wb_rpt.save(str(rpt_path))
    return rpt_path


if facturas_para_reporte:
    rpt = generar_reporte_revision(facturas_para_reporte, OUTPUT_DIR)
    print(f'Reporte de revisión guardado: {rpt}')
    n_nc_nd  = sum(1 for f in facturas_para_reporte if f.get('tipo_doc') in ('NOTA_CREDITO', 'NOTA_DEBITO'))
    n_baja   = sum(1 for f in facturas_para_reporte if f.get('confianza_clasificacion', 0) < 0.70)
    n_media  = sum(1 for f in facturas_para_reporte if 0.70 <= f.get('confianza_clasificacion', 0) < 0.90)
    print(f'  NC/ND (revisión obligatoria): {n_nc_nd}')
    print(f'  Amarillo (<0.70): {n_baja}')
    print(f'  Naranja (0.70-0.89): {n_media}')
else:
    print('No hay facturas para revisión — todas son facturas ordinarias con confianza ≥ 0.90')

Reporte de revisión guardado: ..\data\output\reporte_revision_20260709.xlsx


  NC/ND (revisión obligatoria): 1
  Amarillo (<0.70): 0
  Naranja (0.70-0.89): 0


## 6. Verificación de cuadre Debe = Haber

In [9]:
print('=== VERIFICACIÓN DE CUADRE POR ASIENTO ===')
print('(Solo asientos nuevos generados en este lote)\n')

todas_ok = True

for factura in compras + ventas:
    tipo_doc = factura.get('tipo_doc', 'FACTURA')
    serie    = factura.get('serie', '')
    num      = factura.get('numero', '')
    tipo_op  = factura.get('tipo_operacion')

    base  = round2(factura.get('base_imponible'))
    igv   = round2(factura.get('igv'))
    total = round2(factura.get('total'))

    # Aplicar la misma normalización que los generadores de asiento
    if abs(base + igv - total) > 0.02:
        base = round2(total / 1.18)
        igv  = round2(total - base)

    if tipo_doc in TIPOS_AJUSTE:
        if tipo_op == 'VENTA':
            # NC venta: 40111 DEBE igv + 7041 DEBE base | 12121 HABER total
            debe  = round2(igv + base)
            haber = round2(total)
        else:
            # NC compra: 42121 DEBE total + 79 DEBE base | 40111 HABER igv + 6X HABER base + 9411 HABER base
            debe  = round2(total + base)
            haber = round2(igv + base + base)
    elif tipo_op == 'COMPRA':
        # gasto DEBE base + 40111 DEBE igv + 9411 DEBE base | 42121 HABER total + 79 HABER base
        debe  = round2(base + igv + base)
        haber = round2(total + base)
    else:
        # VENTA: 12121 DEBE total | 40111 HABER igv + 7041 HABER base
        debe  = round2(total)
        haber = round2(igv + base)

    cuadra = abs(debe - haber) < 0.02
    icon   = '✅' if cuadra else '❌'
    print(f'  {icon} {tipo_op} {tipo_doc} {serie}-{num}: Debe={debe:.2f} Haber={haber:.2f}')
    if not cuadra:
        todas_ok = False

print()
if todas_ok:
    print('✅ TODOS LOS ASIENTOS CUADRAN')
else:
    print('❌ HAY ASIENTOS QUE NO CUADRAN — revisar manualmente')


=== VERIFICACIÓN DE CUADRE POR ASIENTO ===
(Solo asientos nuevos generados en este lote)

  ✅ VENTA FACTURA E001-00000030: Debe=8850.00 Haber=8850.00
  ✅ VENTA FACTURA E001-00000031: Debe=1770.00 Haber=1770.00
  ✅ VENTA NOTA_CREDITO E001-00000003: Debe=8850.00 Haber=8850.00

✅ TODOS LOS ASIENTOS CUADRAN
